# Random Forest Classifier for Fake Review Detection

## 1. Introduction
Random Forest is an ensemble learning method that fits a number of decision tree classifiers on various sub-samples of the dataset and uses averaging to improve the predictive accuracy and control over-fitting.

In this notebook, we use the precomputed TF-IDF features to train and optimize a Random Forest model.

---


In [2]:
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Constants
DATA_PATH = '../data/processed/X_tfidf.pkl'
LABEL_PATH = '../data/processed/y.pkl'
MODEL_DIR = '../data/processed'
SEED = 42


## 2. Loading and Splitting Data
We load the precomputed TF-IDF features and labels. We split the data into training (80%) and testing (20%) sets, using **stratification** to maintain the class balance.


In [ ]:
X = joblib.load(DATA_PATH)
y = joblib.load(LABEL_PATH)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Data loaded and split:")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")


## 3. Hyperparameter Tuning
To optimize the Random Forest model, we perform a randomized search over a broad range of hyperparameters:
- **`n_estimators`**: Number of trees in the forest.
- **`max_depth`**: Maximum depth of each tree.
- **`min_samples_split`**: Minimum number of samples required to split a node.
- **`min_samples_leaf`**: Minimum number of samples required at each leaf node.
- **`max_features`**: Number of features to consider when looking for the best split.
- **`bootstrap`**: Whether bootstrap samples are used when building trees.

We use `RandomizedSearchCV` with 5-fold cross-validation and 30 iterations to efficiently find the best combination.


In [ ]:
param_dist = {
    'n_estimators': [200, 300, 500],
    'max_depth': [None, 20, 40, 60],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

print("Starting Randomized Search...")
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1,
    random_state=SEED
)

random_search.fit(X_train, y_train)

best_rf_model = random_search.best_estimator_
print(f"Best Parameters: {random_search.best_params_}")
print(f"Best Cross-Validation Score: {random_search.best_score_:.4f}")


## 4. Evaluation
We evaluate the best model on the test set.


In [ ]:
y_pred = best_rf_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Original (OR)', 'Computer Generated (CG)']))


## 5. Visualizing Results
### 5.1. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['OR', 'CG'], yticklabels=['OR', 'CG'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Random Forest")
plt.show()


### 5.2. Feature Importance
Understanding which words contribute most to the model's decisions.


In [ ]:
# Get feature names from the saved vectorizer
vectorizer = joblib.load(os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))
feature_names = vectorizer.get_feature_names_out()

# Get importances from the best model
importances = best_rf_model.feature_importances_
indices = np.argsort(importances)[-20:]  # Top 20 features

plt.figure(figsize=(10, 8))
plt.title('Top 20 Important Features - Random Forest')
plt.barh(range(len(indices)), importances[indices], color='g', align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()


## 6. Saving the Model

In [ ]:
model_filename = os.path.join(MODEL_DIR, 'rf_model.pkl')
joblib.dump(best_rf_model, model_filename)
print(f"Model saved to: {model_filename}")